[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/surface_metrics/glencoe_hillshade_slope.ipynb)

# Hillshade and slope maps of Glencoe

In this notebook you will:

1. Download a digital elevation model (DEM) of Glencoe in the Scottish Highlands with `lsdtt-fetch-raster`.
2. Compute a hillshade and a slope raster with `lsdtt-surface-metrics`.
3. Make two maps with `lsdviztools3`: a simple hillshade, and slope draped over the hillshade.

The `lsdtt3` programs are command-line tools controlled by small text **parameter files**. We write these files from Python and then run the programs with `!`.

## Setup

### First set up condacolab

This step takes around 2 minutes. **`condacolab.install()` restarts the Colab runtime.** You will see a message that the session crashed: this is expected. Wait for it to reconnect, then carry on with the next cell.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

Now install `pygmt` (GMT is the Generic Mapping Tools). This is the slowest step and takes around a minute.

In [ ]:
!mamba install pygmt

Now get `lsdviztools3`, the plotting package.

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdviztools3-0.1.0-py3-none-any.whl

In [ ]:
!pip install lsdviztools3-0.1.0-py3-none-any.whl

Next, download and unpack the `lsdtt3` command-line programs.

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdtt3-backend-linux-x86_64-core-v0.5.2.tar.gz

In [ ]:
!tar -xzf lsdtt3-backend-linux-x86_64-core-v0.5.2.tar.gz

Tell the system where to find the `lsdtt3` programs and their libraries.

In [ ]:
import os

root = "/content/lsdtt3-backend-linux-x86_64-core-v0.5.2"
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Do NOT set GDAL_DATA / PROJ_DATA / PROJ_LIB here. The lsdtt3 programs find
# their own bundled PROJ and GDAL data, and pointing the Python session at the
# backend's older proj.db breaks rasterio and pyproj.

Check that it works: this should print version information.

In [ ]:
!lsdtt-surface-metrics -v

## Step 1: Download a DEM of Glencoe

`lsdtt-fetch-raster` downloads Copernicus GLO-30 elevation data (30 m resolution, no account needed) for a box given in longitude and latitude.

It also **reprojects** the data into a metric coordinate system. Slope is a change in height divided by a horizontal distance, so both need to be in metres. Glencoe lies in UTM zone 30N (EPSG:32630). The tool would pick this zone automatically, but we set it explicitly.

The output is written to `<write_fname>_DEM.tif`, so here it will be `glencoe_DEM.tif`.

In [ ]:
# Parameter file for lsdtt-fetch-raster
with open("glencoe_fetch.param", "w") as f:
    f.write("write_fname: glencoe\n")
    f.write("west: -5.25\n")
    f.write("east: -4.75\n")
    f.write("south: 56.55\n")
    f.write("north: 56.75\n")
    f.write("dem_source: cop30_aws\n")
    f.write("target_epsg: 32630\n")
    f.write("grid_spacing: 30\n")

!lsdtt-fetch-raster ./ glencoe_fetch.param

In [ ]:
!ls -lh glencoe*

## Step 2: Compute a hillshade and slope

`lsdtt-surface-metrics` calculates many properties of a surface. We ask for two:

- `write_hillshade_raster: true` gives a hillshade, a picture of the landscape lit by a sun in the northwest (azimuth 315°, 45° above the horizon). Output: `glencoe_hillshade.tif`.
- `polyfit_metrics: slope` fits a smooth surface to the elevations around each pixel and computes the topographic gradient from it (dimensionless, metres of drop per metre of horizontal distance, so 0.5 is a 1 in 2 slope, about 27°). `polyfit_lambda_metres` sets the size of the window used for the fit (70 m suits 30 m data). Output: `glencoe_direct_polyfit_slope.tif`.

Note that `read_fname` needs the file extension.

In [ ]:
# Parameter file for lsdtt-surface-metrics
with open("glencoe_metrics.param", "w") as f:
    f.write("read_fname: glencoe_DEM.tif\n")
    f.write("write_fname: glencoe\n")
    f.write("write_hillshade_raster: true\n")
    f.write("hillshade_azimuth: 315\n")
    f.write("hillshade_altitude: 45\n")
    f.write("polyfit_lambda_metres: 70\n")
    f.write("polyfit_metrics: slope\n")

!lsdtt-surface-metrics ./ glencoe_metrics.param

In [ ]:
!ls -lh glencoe*

## Step 3: A simple hillshade map

Now we plot the hillshade with `lsdviztools3`. `render_hillshade` draws a raster and returns a PyGMT figure.

- `shade=False` because our raster is already a hillshade (we don't want to shade it a second time).
- `MapStyle(clean=True)` removes all the map decoration (axes, labels, title) and adds a small black scale bar.
- `coordinates="local"` draws the map in UTM coordinates.

In [ ]:
from IPython.display import Image, display

from lsdviztools3.render.hillshade import render_hillshade
from lsdviztools3.render.channels import render_drape
from lsdviztools3.render.style import MapStyle
from lsdviztools3.render.base import save_figure

hs_style = MapStyle(cmap="gray", transparent=False, clean=True)
fig = render_hillshade(
    "glencoe_hillshade.tif",
    coordinates="local",
    shade=False,
    style=hs_style,
)
save_figure(fig, "glencoe_hillshade.png", style=hs_style)
display(Image("glencoe_hillshade.png", width=700))

## Step 4: Slope draped over the hillshade

`render_drape` draws a gray hillshade and then lays a second raster (here slope) on top in colour. `transparency` controls how much of the hillshade shows through (0 = solid colour, 100 = invisible).

This time we leave `clean` off, so the map has coordinates (in km) on the x and y axes. `colorbar=True` adds a colour bar.

**Question:** where are the steepest slopes? How do they relate to the valleys you can see in the hillshade?

In [ ]:
slope_style = MapStyle(cmap="jet", transparent=False)
fig = render_drape(
    "glencoe_hillshade.tif",                     # gray hillshade base
    "glencoe_direct_polyfit_slope.tif",  # coloured gradient layer
    coordinates="local",
    base_is_hillshade=True,
    transparency=40,
    cmap="jet",
    colorbar=True,
    colorbar_label="Gradient (m/m)",
    colorbar_position="right",
    style=slope_style,
)
save_figure(fig, "glencoe_slope.png", style=slope_style)
display(Image("glencoe_slope.png", width=700))

## Things to try

- Change `hillshade_azimuth` to move the sun (e.g. `45` for a northeast sun) and see how the hillshade changes.
- Change `polyfit_lambda_metres` to `150` and see how the slope map changes.
- Change the bounding box to look at a different mountain area.